In [6]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from myimports import *

from sentence_transformers.util import mine_hard_negatives
from minehardnegativesextended_function import mine_hard_negatives_extended

# 3. Load a dataset to finetune on
train_dataset = load_dataset("json", data_files="../data/trn.json", split="train")
eval_dataset = load_dataset("json", data_files="../data/eval.json", split="train")
test_dataset = load_dataset("json", data_files="../data/tst.json", split="train")


def fixcols(ds):
    ds = ds.rename_column("anchor", "query")
    ds = ds.rename_column("positive", "answer")
    ds = ds.remove_columns(["id"])
    return ds

train_dataset=fixcols(train_dataset)
eval_dataset=fixcols(eval_dataset)
test_dataset=fixcols(test_dataset)

# def dropduplicaterows(ds: "Dataset"):
#     """
#     Remove duplicate rows from a dataset.

#     Args:
#         ds (huggingface dataset): The input dataset.

#     Returns:
#         datasets.Dataset: The dataset with duplicate rows removed.
#     """
#     ds = pd.DataFrame(ds)
#     ds = ds.drop_duplicates()
#     ds = datasets.Dataset.from_pandas(ds, preserve_index=False)
#     return ds

#create a corpus spanning all answer columns
corpus_dataset = concatenate_datasets([train_dataset, eval_dataset, test_dataset])
corpus_dataset=corpus_dataset.remove_columns(["query"])

#load a model to calculate similarities with
model = SentenceTransformer("all-MiniLM-L6-v2")

dataset = mine_hard_negatives_extended(
        dataset=train_dataset,
        model=model,
        corpus=corpus_dataset,
        range_min=10,
        range_max=50,
        max_score=0.8,
        margin=0.1,
        num_negatives=5,
        sampling_strategy="random",
        batch_size=128,
        use_faiss=False,
        verbose=True,
    )

#add a new column that consists of empty lists
new_column = list(range(len(dataset))) 
dataset=dataset.add_column('id',new_column)
dataset=dataset.rename_column( "query","anchor")
dataset=dataset.rename_column( "answer","positive")

#save it and reload it
dataset.to_json(f'../data/trn_with_hard_negatives_HF_new.json',orient='records',lines=True)
load_dataset("json", data_files="../data/trn_with_hard_negatives_HF_new.json", split="train")
dataset

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Corpus length: 3348, number unique in corpus= 3348
Queries length: 3348, number unique in queries= 3289


Batches: 100%|██████████| 27/27 [00:00<00:00, 36.42it/s]


Metric       Positive       Negative     Difference
Count           3,348          9,132          9,132
Mean           0.5721         0.4183         0.2418
Median         0.5761         0.4190         0.2226
Std            0.1487         0.0829         0.0998
Min           -0.0570         0.1325         0.1043
25%            0.4781         0.3632         0.1585
50%            0.5762         0.4190         0.2226
75%            0.6788         0.4732         0.3046
Max            0.9780         0.7323         0.6440
Skipped 88949 potential negatives (52.09%) due to the margin of 0.1.
Skipped 5 potential negatives (0.01%) due to the maximum score of 0.8.
Could not find enough negatives for 7608 samples (45.45%). Consider adjusting the range_max, range_min, margin and max_score parameters if you'd like to find more valid negatives.


Creating json from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 108.62ba/s]
Generating train split: 9132 examples [00:00, 228426.84 examples/s]


Dataset({
    features: ['anchor', 'positive', 'negative', 'id'],
    num_rows: 9132
})

In [ ]:
# hnm(corpus_dataset1)
# hnm(corpus_dataset3)
# hnm(corpus_dataset3)

Corpus length: 3348, number unique in corpus= 3348
Queries length: 3348, number unique in queries= 3289


Batches: 100%|██████████| 27/27 [00:00<00:00, 35.97it/s]


Metric       Positive       Negative     Difference
Count           3,348          9,132          9,132
Mean           0.5721         0.4182         0.2419
Median         0.5761         0.4184         0.2236
Std            0.1487         0.0827         0.0997
Min           -0.0570         0.1264         0.1043
25%            0.4781         0.3642         0.1585
50%            0.5762         0.4184         0.2236
75%            0.6788         0.4732         0.3042
Max            0.9780         0.7309         0.6628
Skipped 88949 potential negatives (52.09%) due to the margin of 0.1.
Skipped 5 potential negatives (0.01%) due to the maximum score of 0.8.
Could not find enough negatives for 7608 samples (45.45%). Consider adjusting the range_max, range_min, margin and max_score parameters if you'd like to find more valid negatives.


Dataset({
    features: ['query', 'answer', 'negative'],
    num_rows: 9132
})

In [7]:
#the existing way
dataset = mine_hard_negatives(
    dataset=train_dataset,
    model=model,
    range_min=10,
    range_max=50,
    max_score=0.8,
    margin=0.1,
    num_negatives=5,
    sampling_strategy="random",
    batch_size=128,
    use_faiss=False,
    verbose=True,
 )
#add a new column that consists of empty lists
new_column = list(range(len(dataset))) 
dataset=dataset.add_column('id',new_column)
dataset=dataset.rename_column( "query","anchor")
dataset=dataset.rename_column( "answer","positive")

#save it and reload it
dataset.to_json(f'../data/trn_with_hard_negatives_HF_old.json',orient='records',lines=True)
load_dataset("json", data_files="../data/trn_with_hard_negatives_HF_old.json", split="train")
dataset

Batches: 100%|██████████| 27/27 [00:00<00:00, 78.36it/s]


Metric       Positive       Negative     Difference
Count           3,348          9,132          9,132
Mean           0.5721         0.4183         0.2418
Median         0.5761         0.4194         0.2242
Std            0.1487         0.0826         0.0994
Min           -0.0570         0.1363         0.1040
25%            0.4781         0.3638         0.1585
50%            0.5762         0.4194         0.2243
75%            0.6788         0.4734         0.3039
Max            0.9780         0.7332         0.6640
Skipped 88949 potential negatives (52.09%) due to the margin of 0.1.
Skipped 5 potential negatives (0.01%) due to the maximum score of 0.8.
Could not find enough negatives for 7608 samples (45.45%). Consider adjusting the range_max, range_min, margin and max_score parameters if you'd like to find more valid negatives.


Creating json from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 92.84ba/s]
Generating train split: 9132 examples [00:00, 392518.87 examples/s]


Dataset({
    features: ['anchor', 'positive', 'negative', 'id'],
    num_rows: 9132
})